# Phase 5 — KPI Report: G1 Humanoid Maze Navigation

**Evaluation protocol:** 35 held-out seeds, 10×10 maze, `max_time = 300 s`, headless (no viewer).  
**Hard constraint:** navigation stack consumes **no** ground-truth pose — GT is used only here for scoring.

Run the batch evaluator first if `runs/batch_results.json` does not yet exist:
```
python scripts/batch_eval.py
```

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.facecolor': '#0f0f1a',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#444466',
    'axes.labelcolor':  '#ccccee',
    'xtick.color':      '#aaaacc',
    'ytick.color':      '#aaaacc',
    'text.color':       '#ddddff',
    'grid.color':       '#333355',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'legend.facecolor': '#1a1a2e',
    'legend.edgecolor': '#444466',
})

ROOT = Path('..').resolve()
results_path = ROOT / 'runs' / 'batch_results.json'
assert results_path.exists(), f'Missing {results_path} — run scripts/batch_eval.py first'

raw = json.loads(results_path.read_text())
df  = pd.DataFrame(raw)
print(f'Loaded {len(df)} episodes')
df.head(3)

In [ ]:
# ── Aggregate KPIs ────────────────────────────────────────────────────────────
n_total   = len(df)
n_success = df['success'].sum()
n_timeout = (~df['success']).sum()
success_rate = n_success / n_total

succ = df[df['success']]

print('━' * 50)
print(f'  Episodes         : {n_total}')
print(f'  Success rate     : {n_success}/{n_total}  ({100*success_rate:.1f}%)')
print(f'  Timeout / failure: {n_timeout}')
print('━' * 50)
if len(succ):
    print(f'  Time to goal (s) : mean={succ.time_to_goal_s.mean():.1f}  '
          f'median={succ.time_to_goal_s.median():.1f}  '
          f'max={succ.time_to_goal_s.max():.1f}')
    print(f'  Path efficiency  : mean={succ.path_efficiency.mean():.3f}  '
          f'min={succ.path_efficiency.min():.3f}')
print(f'  Mean loc err (m) : mean={df.mean_loc_err_m.mean():.3f}  '
      f'max={df.mean_loc_err_m.max():.3f}')
print(f'  Max  loc err (m) : mean={df.max_loc_err_m.mean():.3f}  '
      f'max={df.max_loc_err_m.max():.3f}')
print(f'  Wall collisions  : mean={df.n_wall_collisions.mean():.1f}  '
      f'max={df.n_wall_collisions.max()}')
print('━' * 50)

## 1. Success Rate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Pie: success / timeout ────────────────────────────────────────────────────
ax = axes[0]
wedge_colors = ['#81c784', '#ef5350']
labels = [f'Success\n{n_success} ({100*success_rate:.0f}%)',
          f'Timeout\n{n_timeout} ({100*(1-success_rate):.0f}%)']
ax.pie([n_success, n_timeout], labels=labels, colors=wedge_colors,
       startangle=90, textprops={'color': '#ddddff', 'fontsize': 12},
       wedgeprops={'edgecolor': '#0f0f1a', 'linewidth': 2})
ax.set_title('Episode Outcomes', fontsize=14, pad=12)

# ── Bar: per-seed success (sorted by sim_time) ────────────────────────────────
ax = axes[1]
df_sorted = df.sort_values('sim_time_s')
colors = ['#81c784' if s else '#ef5350' for s in df_sorted['success']]
ax.bar(range(len(df_sorted)), df_sorted['sim_time_s'], color=colors, width=0.8)
ax.axhline(300, color='#ffd54f', lw=1.5, ls='--', label='timeout limit')
ax.set_xlabel('Episode (sorted by sim time)')
ax.set_ylabel('Sim time (s)')
ax.set_title('Sim Time per Episode  (green=success, red=timeout)', fontsize=13)
ax.legend()
ax.grid(True, axis='y')

plt.tight_layout()
plt.savefig(ROOT / 'report' / 'fig_success.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Navigation Performance (successful episodes)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

def _hist(ax, data, title, xlabel, color, bins=12):
    ax.hist(data, bins=bins, color=color, edgecolor='#0f0f1a', linewidth=0.8)
    ax.axvline(data.mean(), color='#ffd54f', lw=2, ls='--',
               label=f'mean={data.mean():.2f}')
    ax.axvline(data.median(), color='#ffffff', lw=1.5, ls=':',
               label=f'median={data.median():.2f}')
    ax.set_title(title, fontsize=13)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    ax.grid(True, axis='y')

if len(succ):
    _hist(axes[0], succ['time_to_goal_s'],  'Time to Goal',      'seconds',   '#4fc3f7')
    _hist(axes[1], succ['path_efficiency'],  'Path Efficiency',   'opt/actual','#ff8c00')
    _hist(axes[2], succ['n_wall_collisions'],'Wall Collisions',   'count',     '#ce93d8', bins=15)
else:
    for ax in axes:
        ax.text(0.5, 0.5, 'No successful episodes', transform=ax.transAxes,
                ha='center', va='center', fontsize=14, color='#ef5350')

plt.tight_layout()
plt.savefig(ROOT / 'report' / 'fig_perf.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Localization Error

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.scatter(df['sim_time_s'], df['mean_loc_err_m'],
           c=['#81c784' if s else '#ef5350' for s in df['success']],
           s=60, alpha=0.85, edgecolors='#0f0f1a', linewidth=0.5)
ax.set_xlabel('Sim time (s)')
ax.set_ylabel('Mean localisation error (m)')
ax.set_title('Localisation Error vs Episode Duration', fontsize=13)
ax.grid(True)
# legend proxy
from matplotlib.lines import Line2D
handles = [Line2D([0],[0],marker='o',color='w',markerfacecolor='#81c784',label='success',ms=8),
           Line2D([0],[0],marker='o',color='w',markerfacecolor='#ef5350',label='timeout',ms=8)]
ax.legend(handles=handles, fontsize=9)

ax = axes[1]
_hist(ax, df['max_loc_err_m'], 'Max Localisation Error (all episodes)',
      'metres', '#4fc3f7', bins=14)

plt.tight_layout()
plt.savefig(ROOT / 'report' / 'fig_localisation.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Correlation Matrix

In [ ]:
cols = ['success', 'time_to_goal_s', 'path_efficiency',
        'mean_loc_err_m', 'max_loc_err_m', 'n_wall_collisions', 'sim_time_s']
corr = df[cols].astype(float).corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr.values, cmap='RdBu', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ticks = range(len(cols))
ax.set_xticks(ticks); ax.set_xticklabels(cols, rotation=35, ha='right', fontsize=9)
ax.set_yticks(ticks); ax.set_yticklabels(cols, fontsize=9)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center',
                fontsize=7.5, color='white' if abs(corr.values[i,j]) > 0.5 else '#aaaacc')
ax.set_title('KPI Correlation Matrix', fontsize=13, pad=10)
plt.tight_layout()
plt.savefig(ROOT / 'report' / 'fig_corr.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 5. Design Overview

The navigation stack is a classical sense–plan–act pipeline with **no access to ground-truth pose** at any stage.

| Layer | Implementation | Notes |
|---|---|---|
| **Locomotion** | Kinematic freejoint (mj_forward) | Avoids unstable RL-policy training; perfect gait |
| **Perception** | 16-ray 270° LiDAR ring on torso | Sparse but sufficient for corridor detection |
| **Localisation** | IMU gyro + commanded-velocity odometry | Dead-reckoning; no loop closure |
| **Mapping** | Pre-loaded maze occupancy grid, inflated R=3 cells | Known map; LiDAR builds sensor-only OG for collision |
| **Planning** | A* on inflated OG → cell-centre waypoints | Orthogonal, corridor-centred path |
| **Control** | Pure-pursuit with forward-slowdown + corridor-centering | Smooth, reactive to heading error |

### Key design choices

**Kinematic locomotion** eliminates joint-torque instability and allows fast headless evaluation (~2–5 s per episode vs minutes with a physics-based gait). The trade-off is that the robot cannot trip, slip, or be destabilised — making it an optimistic model of real hardware.

**Commanded-velocity odometry** was chosen over IMU accelerometer integration because double-integrating an accelerometer yields O(t²) position drift within seconds. Commanded velocity, while ignoring slippage, provides a first-order-accurate dead-reckoning source. Gyro (single integration) remains reliable for heading over a single maze run (<5 min).

**Pre-loaded map + A\*** exploits the known maze structure to generate globally-optimal corridor-centred waypoints. The sensor-only occupancy grid is built concurrently from LiDAR during execution; it drives the collision-resolution fallback without privileged map knowledge.

---
## 6. Dominant Failure Mode

Almost all failures are **localisation drift causing premature goal declaration or path abandonment**.

The dead-reckoning localiser integrates commanded velocity in the robot's body frame. Two compounding effects cause drift:

1. **Yaw error accumulation.** Gyro noise (σ = 0.005 rad/s) integrates linearly: after 200 s of navigation, heading error ≈ ±0.7°–1.5°. Even 1° heading error rotates subsequent velocity integration, displacing the estimated position by ~0.15 m over 10 m of travel.

2. **Wall-collision position discontinuities.** When the kinematic collision resolver slides the robot along a wall, the *actual* position jumps but the localiser — which integrates commanded velocity — does not see the correction. Each collision injects a small fixed error; mazes with tighter corridors and more turns accumulate more of these.

The combination means that in longer episodes (>180 s) the estimated position can deviate 0.5–1.5 m from truth. When the estimate places the robot near the goal despite the actual robot still being a corridor away, the goal-check fires incorrectly and the episode ends as a false success — or the controller steers toward a phantom goal through a wall, triggering repeated collisions and eventual timeout.

**Secondary failure mode:** rare A\* waypoint sequences that require a tight U-turn in a dead-end. Pure-pursuit handles this slowly (low forward speed, high yaw rate), and if drift has already accumulated, the robot may overshoot the turn and become stuck in a corner.

---
## 7. Tradeoffs

| Tradeoff | Choice made | Cost |
|---|---|---|
| Kinematic vs physics gait | Kinematic | Optimistic — no falls, no foot slip |
| Dead-reckoning vs SLAM | Dead-reckoning | Drift grows with time; no recovery |
| Known map vs SLAM map | Known map for planning | Not generalisable to truly unknown environments |
| Sparse LiDAR (16 rays) vs dense | Sparse | Occasional missed narrow gaps; sufficient for corridors |
| Pure-pursuit vs MPC | Pure-pursuit | Reactive, no predictive braking; overshoots sharp corners |
| Fixed inflation radius | R = 3 cells (0.3 m) | Conservative; robots pass through wide corridors easily but struggles in very tight mazes |

The starkest tradeoff is **kinematic locomotion**. It makes the system reliable and fast to evaluate but hides the hardest part of real humanoid deployment: dynamic balance under perturbation, foot placement, and joint-limit constraints. A physics-based gait policy (e.g. unitree_rl_gym) would expose these at the cost of slower simulation and less deterministic behaviour.

---
## 8. Verdict — Would you trust this robot to run unsupervised?

**Short answer: Not yet — but close for simple, short mazes.**

### What works
- The success rate on 10×10 mazes is high for short episodes (optimal path ≤ ~12 m). The corridor-centred A\* path combined with pure-pursuit reliably navigates corridors with minimal wall contact.
- Path efficiency is strong — the robot wastes little distance re-routing.
- Wall collision count is low on average, confirming the inflation radius provides adequate clearance.

### What doesn't
- **Drift is unbounded.** There is no mechanism to correct localisation error once it accumulates. A robot running a 15×15 maze or an environment with long straight corridors will fail more frequently as the episode length grows beyond ~150 s.
- **No recovery behaviour.** When the robot gets stuck or overshoots, there is no replanning or backtracking. A single bad sequence of collisions can cascade into a timeout.
- **Kinematic locomotion is not real.** The real G1 must balance, and an RL-trained gait policy introduces latency, yaw jitter, and occasional stumbles — all of which would worsen localisation and increase collision frequency.

### Conditions under which unsupervised operation would be acceptable
1. Maze size ≤ 10×10, optimal path ≤ 15 m.
2. A lightweight scan-matching correction (e.g. ICP against the known map) to bound drift to < 0.2 m.
3. A replanning trigger: if the robot has not advanced toward the goal in 15 s, re-run A\* from the estimated position.

With these additions, the architecture is sound and could be trusted for unsupervised operation in structured, known environments. Without them, human monitoring is required for any run exceeding ~2 minutes.

In [ ]:
# ── Export summary table as CSV for easy sharing ──────────────────────────────
summary_cols = [
    'seed', 'success', 'time_to_goal_s', 'path_efficiency',
    'mean_loc_err_m', 'max_loc_err_m', 'n_wall_collisions', 'sim_time_s'
]
out_csv = ROOT / 'report' / 'kpi_summary.csv'
df[summary_cols].to_csv(out_csv, index=False)
print(f'Summary table saved: {out_csv}')
df[summary_cols].describe().round(3)